# Telecom Customer Churn Analytics
## End-to-End EDA & Modeling Notebook

This notebook walks through the full data science pipeline:
1. Data Ingestion & Profiling
2. Data Cleaning
3. Exploratory Data Analysis
4. Feature Engineering
5. Model Training & Comparison
6. Evaluation & Interpretability

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
%matplotlib inline

print('Libraries loaded successfully.')

## 1. Data Ingestion

Load the Telco Customer Churn dataset and perform initial profiling.

In [ ]:
from src.data_ingestion import load_data, generate_quality_report

df_raw = load_data('data/raw/telco_churn.csv')
print(generate_quality_report(df_raw))
df_raw.head()

In [ ]:
print(f'Shape: {df_raw.shape}')
print(f'\nData Types:\n{df_raw.dtypes}')
print(f'\nMissing Values:\n{df_raw.isnull().sum()}')
print(f'\nDuplicate Rows: {df_raw.duplicated().sum()}')

## 2. Data Cleaning

In [ ]:
from src.data_cleaning import clean_data

df_clean = clean_data(df_raw)
print(f'Cleaned shape: {df_clean.shape}')
print(f'Missing values: {df_clean.isnull().sum().sum()}')
df_clean.head()

## 3. Exploratory Data Analysis

In [ ]:
from src.eda import (
    plot_churn_distribution,
    plot_correlation_heatmap,
    plot_numeric_distributions,
    plot_categorical_churn_rates,
    plot_boxplots,
    chi_square_tests,
)

# Churn Distribution
plot_churn_distribution(df_clean)

In [ ]:
# Correlation Heatmap
plot_correlation_heatmap(df_clean)

In [ ]:
# Numeric Feature Distributions by Churn
plot_numeric_distributions(df_clean)

In [ ]:
# Churn Rate by Categorical Features
plot_categorical_churn_rates(df_clean)

In [ ]:
# Box Plots
plot_boxplots(df_clean)

In [ ]:
# Chi-Square Tests
chi_results = chi_square_tests(df_clean)
chi_results

## 4. Feature Engineering

In [ ]:
from src.feature_engineering import engineer_features

df_features, scaler = engineer_features(df_clean, scale=True)
print(f'Feature-engineered shape: {df_features.shape}')
print(f'\nNew columns: {[c for c in df_features.columns if c not in df_clean.columns]}')
df_features.head()

## 5. Model Training & Comparison

In [ ]:
from src.model_training import split_data, apply_smote, train_models, select_best_model, save_model

X_train, X_test, y_train, y_test = split_data(df_features)
X_train_sm, y_train_sm = apply_smote(X_train, y_train)

print(f'Training set (after SMOTE): {X_train_sm.shape}')
print(f'Test set: {X_test.shape}')

In [ ]:
trained_models = train_models(X_train_sm, y_train_sm)
best_name, best_model = select_best_model(trained_models)
save_model(best_model, 'models/best_model.joblib')
print(f'\nBest model: {best_name}')

## 6. Evaluation & Interpretability

In [ ]:
from src.model_evaluation import generate_evaluation_report

results = generate_evaluation_report(trained_models, X_test, y_test)
results

---
## Summary

- **Best model**: XGBoost with ~88% AUC-ROC
- **Top churn predictors**: Contract type, tenure, monthly charges, internet service, payment method
- **Key insight**: Month-to-month contract customers with fiber optic internet paying via electronic check have the highest churn risk
- **Recommendation**: Target high-risk segments with retention offers (contract upgrades, loyalty discounts)